# 06 · Robustness synthesis, Monte Carlo error and the pilot as antecedent

**Purpose.** Bring together, for every reference model, the quantities that decide the paper: global structure (notebook 03), year-by-year similarity (notebook 04), uniform-difference strength and the relative improvements in model fit (notebook 05), and the pilot's two-window comparison. It adds two checks:

1. **Monte Carlo error.** The year-vs-rest correlations are recomputed with the expected counts of each of the four chains separately; the spread across chains measures simulation error, not sampling uncertainty.
2. **Losses.** For each reference, the sample retained and the mobility of the chains (acceptance, share of fields changed, fields in strata where no swap is possible).

No new specification is introduced here.

The archive retains the original numerical column names. In the summary, `frequency_share_*` denotes relative model-fit improvement. The S11 calculation uses finite, signed residuals.

In [1]:
import os
for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(k, "2")
import json, itertools
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
TAB = ROOT / "results" / "tables"; AGG = ROOT / "results" / "aggregates"
def display_and_export_table(df, name, index=False, float_format="%.4f"):
    display(df)
    df.to_csv(TAB / f"{name}.csv", index=index)
    df.to_latex(TAB / f"{name}.tex", index=index, escape=True, float_format=lambda x: float_format % x)
    return df
PAIRS = np.array(list(itertools.combinations(range(1, 36), 2)))
YEARS = np.arange(2008, 2025); T = len(YEARS)
SPECS = ["N1", "N2", "N2_clean", "N2_allmedia", "N2_order", "N1_k2"]
elig = np.load(AGG / "eligible_primary.npy")
lr = lambda O, E: np.log2((O + .5) / (E + .5))

## 1. Monte Carlo error across chains

In [2]:
rows = []
for name in SPECS:
    z = np.load(AGG / f"null_{name}.npz")
    C = z["counts"].astype(float); ch = z["chain"]; Oy = z["observed_year"].astype(float)
    sd_all = C.std(0, ddof=1); tot = C.sum(1)
    per_chain = []
    for c in np.unique(ch):
        Ey = C[ch == c].mean(0); r = []
        for ti in range(T):
            Er = Ey.sum(0) - Ey[ti]; sdr = (tot - C[:, ti, :]).std(0, ddof=1)
            m = elig & (Ey[ti] >= 5) & (Er >= 5) & (sd_all[ti] > 0) & (sdr > 0)
            r.append(pearsonr(lr(Oy[ti], Ey[ti])[m], lr(Oy.sum(0) - Oy[ti], Er)[m])[0] if m.sum() >= 3 else np.nan)
        per_chain.append(r)
    per_chain = np.array(per_chain)
    for ti, yr in enumerate(YEARS):
        rows.append({"reference": name, "year": yr, "pearson_chain_min": np.nanmin(per_chain[:, ti]),
                     "pearson_chain_max": np.nanmax(per_chain[:, ti]), "pearson_chain_range": np.nanmax(per_chain[:, ti]) - np.nanmin(per_chain[:, ti])})
mc = display_and_export_table(pd.DataFrame(rows), "monte_carlo_error_year_vs_rest")
mc.groupby("reference").pearson_chain_range.agg(["median", "max"]).round(4)

,median,max
reference,,
N1,0.0045,0.0193
N1_k2,0.0054,0.0174
N2,0.0044,0.1015
N2_allmedia,0.0095,0.0820
N2_clean,0.0082,0.2012
N2_order,0.0097,0.0739


## 2. One row per reference: the decisive quantities

In [3]:
glob = pd.read_csv(TAB / "global_structure_tests.csv").set_index("reference")
yvr = pd.read_csv(TAB / "annual_year_vs_rest.csv")
pilot = pd.read_csv(TAB / "pilot_windows_reproduced.csv").set_index("reference")
mcmc = pd.read_csv(TAB / "mcmc_diagnostics.csv").set_index("spec")
dec = pd.read_csv(TAB / "loglinear_decomposition.csv").set_index("reference")
het = pd.read_csv(TAB / "heterogeneity_summary.csv").set_index("reference")
calib = pd.read_csv(TAB / "annual_calibration.csv")
phit = pd.read_csv(TAB / "unidiff_phi_by_year.csv")
rows = []
for name in SPECS:
    y = yvr[yvr.reference == name]
    row = {"reference": name, "actions": int(mcmc.loc[name, "actions"]), "multi_tactic_actions": int(mcmc.loc[name, "multi_tactic_actions"]),
           "acceptance": mcmc.loc[name, "acceptance_mean"], "changed_fields": mcmc.loc[name, "changed_fields_mean"],
           "fields_in_fixed_strata": mcmc.loc[name, "fields_in_fixed_strata"], "rhat_max": mcmc.loc[name, "rhat_max"],
           "evaluable_pairs_full_period": int(glob.loc[name, "evaluable_pairs"]), "global_excess_ratio": glob.loc[name, "excess_ratio"],
           "yvr_pearson_median": y.pearson.median(), "yvr_pearson_min": y.pearson.min(), "yvr_year_min": int(y.loc[y.pearson.idxmin(), "year"]),
           "yvr_strong_sign_median": y.strong_sign_agreement.median(), "yvr_pairs_median": y.pairs.median(),
           "pilot_windows_pearson": pilot.loc[name, "pearson"], "pilot_published": pilot.loc[name, "pilot_published_pearson"]}
    if name in calib.reference.unique():
        c = calib[calib.reference == name]
        row.update(disattenuated_median=c.disattenuated.median(), disattenuated_min=c.disattenuated.min(),
                   years_detectable_departure=int(c.detectable_departure.sum()),
                   years_departure_list=",".join(str(v) for v in c.loc[c.detectable_departure, "year"]))
    if name in dec.index:
        row.update(frequency_share_raw=dec.loc[name, "frequency_share_raw"], frequency_share_dispersion=dec.loc[name, "frequency_share_dispersion_floor"],
                   frequency_share_permutation=dec.loc[name, "frequency_share_permutation_floor"], share_structure_constant=dec.loc[name, "share_structure_constant"],
                   phi_min=phit[phit.reference == name].phi.min(), phi_max=phit[phit.reference == name].phi.max())
    if name in het.index:
        row.update(pooled_I2=het.loc[name, "pooled_I2"], median_I2=het.loc[name, "median_I2"])
    rows.append(row)
robust = display_and_export_table(pd.DataFrame(rows), "robustness_summary")
robust.set_index("reference").T

reference,N1,N2,N2_clean,N2_allmedia,N2_order,N1_k2
actions,27488,27488,23559,34118,27488,8981
multi_tactic_actions,13405,13405,6666,15863,13405,8981
acceptance,0.620255,0.584209,0.614812,0.597585,0.382012,0.554476
changed_fields,0.871467,0.738566,0.730035,0.747832,0.686546,0.836808
fields_in_fixed_strata,0.0,0.00896,0.011802,0.00704,0.00896,0.0
rhat_max,1.013231,1.011511,1.009244,1.007844,1.013209,1.007306
evaluable_pairs_full_period,291,274,209,280,272,142
global_excess_ratio,93.601083,49.716001,37.083273,55.695549,38.653124,51.247772
yvr_pearson_median,0.845288,0.832855,0.871918,0.841709,0.815327,0.848068
yvr_pearson_min,0.750428,0.697198,0.762689,0.736038,0.657368,0.688083


## 3. Which pairs carry the departures of the flagged years (exploratory)

For the years whose departure from the common structure exceeds within-year noise under N2 (Holm-adjusted $p<0.05$ in notebook 05), we list the cells with the largest Pearson residuals against the structure of the other years, $(O_{ht}-\mu_{ht})/\sqrt{\mu_{ht}}$ (cells with $E_{ht}\ge2$). The list describes where a year differs; it is not a test and selects cells after the fact.

In [4]:

Ftab = pd.read_csv(TAB / "annual_departure_F.csv")
flag = Ftab[(Ftab.reference == "N2") & (Ftab.p_holm < 0.05)].year.tolist()
z = np.load(AGG / "null_N2.npz"); H = np.where(elig)[0]
O = z["observed_year"][:, H].astype(float); E = z["counts"].mean(0)[:, H]
dic = pd.read_csv(TAB / "tactic_dictionary.csv").set_index("code")
def fit_M1(O, E, tol=1e-10, max_iter=5000):
    a = np.ones((O.shape[0], 1)); b = np.ones((1, O.shape[1]))
    for it in range(max_iter):
        nb_, db_ = O.sum(0, keepdims=True), (E * a).sum(0, keepdims=True)
        b_new = np.divide(nb_, db_, out=np.zeros_like(nb_, dtype=float), where=db_ > 0)
        na_, da_ = O.sum(1, keepdims=True), (E * b_new).sum(1, keepdims=True)
        a_new = np.divide(na_, da_, out=np.ones_like(na_, dtype=float), where=da_ > 0)
        done = np.nanmax(np.abs(a_new - a)) < tol and np.nanmax(np.abs(b_new - b)) < tol
        a, b = a_new, b_new
        if done:
            break
    return E * a * b, a.ravel(), b.ravel()
rows = []
for yr in flag:
    ti = yr - 2008; others = [s for s in range(T) if s != ti]
    _, _, b_o = fit_M1(O[others], E[others])
    cells = (E[ti] >= 2) & (b_o > 0)
    mu = E[ti] * b_o; mu = mu * O[ti, cells].sum() / mu[cells].sum()
    res = np.where(cells, (O[ti] - mu) / np.sqrt(np.where(mu > 0, mu, 1)), np.nan)
    # Only finite, eligible residuals are candidates. np.argsort otherwise ranks NaNs last,
    # and reversing the order mistakenly labels unscorable cells as the largest excesses.
    valid = np.flatnonzero(np.isfinite(res))
    order = valid[np.argsort(res[valid])]
    for kind, idx in [("more_than_common_structure", [i for i in order[::-1] if res[i] > 0][:8]),
                      ("fewer_than_common_structure", [i for i in order if res[i] < 0][:8])]:
        for i in idx:
            h = H[i]
            rows.append({"year": yr, "direction": kind, "code_i": PAIRS[h, 0], "code_j": PAIRS[h, 1],
                         "label_i": dic.loc[PAIRS[h, 0], "label_en"], "label_j": dic.loc[PAIRS[h, 1], "label_en"],
                         "observed": int(O[ti, i]), "predicted_common_structure": mu[i], "ratio": O[ti, i] / mu[i], "pearson_residual": res[i]})
dep = display_and_export_table(pd.DataFrame(rows), "departure_pairs_flagged_years_N2")
print("flagged years (N2):", flag)

flagged years (N2): [2016, 2019, 2020, 2021]


,year,direction,code_i,code_j,label_i,label_j,observed,predicted_common_structure,ratio,pearson_residual
0,2016,more_than_common_structure,2,19,Static demonstration,Building occupation,30,12.149460,2.469246,5.121213
1,2016,more_than_common_structure,7,10,Press conference or statement,Chants or slogans,11,3.867078,2.844525,3.627238
2,2016,more_than_common_structure,2,22,Static demonstration,Roadblock or barricade,94,65.826526,1.427996,3.472481
3,2016,more_than_common_structure,1,11,March,Boycott,7,2.078664,3.367548,3.413429
4,2016,more_than_common_structure,18,27,Strike or stoppage,Arson,1,0.074679,13.390720,3.386059
...,...,...,...,...,...,...,...,...,...,...
59,2021,fewer_than_common_structure,2,10,Static demonstration,Chants or slogans,371,411.645755,0.901260,-2.003334
60,2021,fewer_than_common_structure,22,33,Roadblock or barricade,Attack on police or security forces,86,106.548023,0.807148,-1.990661
61,2021,fewer_than_common_structure,2,34,Static demonstration,Attack on bystanders,1,5.670879,0.176340,-1.961433
62,2021,fewer_than_common_structure,27,29,Arson,Looting,2,6.408167,0.312102,-1.741370
